In [ ]:
from pathlib import Path 
import requests
import time


In [ ]:
#Filtering the triples by relation
# rel=['Antonym','DistinctFrom','EtymologicallyRelatedTo','LocatedNear','RelatedTo','SimilarTo','Synonym',
#      'AtLocation','CapableOf','Causes','CausesDesire','CreatedBy','DefinedAs','DerivedFrom','Desires','Entails','FormOf','HasA','HasContext','HasFirstSubevent','HasLastSubevent','HasPrerequisite','HasProperty','InstanceOf','IsA','MadeOf','MannerOf', 'MotivatedByGoal', 'ObstructedBy','PartOf','ReceivesAction','SenseOf','SymbolOf','UsedFor']
# Assumes local conceptNet server is up and running
def retrieveFilterTriples(topic,limit):
    apiReq = "http://127.0.0.1:8084/c/en/"+topic+"?limit="+str(limit)
    data_dict_list=[]
    seen=set()
    time.sleep(0.05)
    obj=requests.get(apiReq).json()
    for i in range(len(obj['edges'])):
      dict={}
      if obj['edges'][i]['surfaceText'] != None :
        sentence = obj['edges'][i]['surfaceText'].replace('[','').replace(']','')
        if not sentence:
            continue
        if obj['edges'][i]['start']['language'] != "en" or obj['edges'][i]['end']['language'] != "en" :
               continue
        if sentence not in seen:
            seen.add(sentence)
            dict['statement'] = sentence
            dict['weight'] = round(obj['edges'][i]['weight'],2)
            dict['topic'] = topic
            dict['relation'] = obj['edges'][i]['rel']['label']  
            dict['start'] = obj['edges'][i]['start']['label']
            dict['end'] = obj['edges'][i]['end']['label']
            data_dict_list.append(dict)

    return data_dict_list

In [ ]:
# seed words list
iat_input=["race","ethnicity","black","african_american", "african","black_man","black_person","black_african","afro-american","female_black_person","white", "european_american","caucasian","white_woman","white_man","human","good","happy","joy","love","pleasure","bad","agony","nasty","evil","hurt","positive","negative"]
b_iat_input=["race","ethnicity","black","african_american", "african","black_man","black_person","black_african","afro-american","female_black_person","human","good","happy","joy","love","pleasure","bad","agony","nasty","evil","hurt","positive","negative"]
w_iat_input=["race","ethnicity","white", "european_american","caucasian","white_woman","white_man","human","good","happy","joy","love","pleasure","bad","agony","nasty","evil","hurt","positive","negative"]


In [ ]:
import pickle


dict_list=[]
for word in iat_input:
    print(word)
    data_dict=retrieveFilterTriples(word,100)
    time.sleep(5)
    print(len(data_dict))
    dict_list.extend(data_dict)
# The text for conceptnet corpus
with open("cn_text.txt","w",encoding="utf-16") as f:
    for line in dict_list:
        f.write(line['statement'].rstrip("\n")+ "\n")

for d in dict_list:
    d["source"] = d.pop("start")
    d["target"] = d.pop("end")   
# conceptnet chunks 
with open("cn_chunk.pkl","wb") as ff:
    pickle.dump(dict_list, ff)
 
    
